## Top autores con más citas

In [25]:
import pandas as pd
import re
from pathlib import Path


In [3]:
df = pd.read_csv("publicaciones_clasificadas.csv")

In [ ]:

#Ruta a la carpeta con los CSVs por autor
carpeta_autores = Path("pubs_autor")

#Leer todos los CSVs que empiezan con productividad_au
archivos = sorted(carpeta_autores.glob("articulos_*.csv"))

# Cargar cada CSV y unirlos en un solo DataFrame
frames = [pd.read_csv(archivo) for archivo in archivos if archivo.is_file()]
df_autores = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

if not df_autores.empty:
    autores_resumen = (
        df_autores.groupby("Scopus author ID", as_index=False)
        .agg(
            Autor=("Autor", "first"),
            Citas=("Citations", "sum"),
        )
        .sort_values("Citas", ascending=False)
    )
else:
    print("No se encontraron archivos CSV en la carpeta autores")

In [15]:
autores_resumen.head()


,Scopus author ID,Autor,Citas
3842,57216482927,"Cárdenas, Rosario",126656.0
954,35373585600,"Borges, Guilherme Luiz Guimaraes",40536.0
2030,56509988200,"Gorini, Giuseppe",12136.0
543,10839525900,"Orozco, Ricardo",6629.0
1318,55399527300,"Álvarez-Ramírez, José J.",6147.0


In [18]:
lista_id_autores = autores_resumen['Scopus author ID'].head(5).tolist()

lista_id_autores

[57216482927, 35373585600, 56509988200, 10839525900, 55399527300]

## Top autores con más indice de colaboracion en las 4 categorias

In [ ]:

carpeta_productividad = Path("productividad_autor")
archivos_prod = sorted(carpeta_productividad.glob("productividad_au*.csv"))

frames_prod = []
for archivo in archivos_prod:
    if archivo.is_file():
        df_temp = pd.read_csv(archivo)
        match = re.search(r'au(\d+)', archivo.stem)
        df_temp['Scopus author ID'] = match.group(1)
        frames_prod.append(df_temp)

df_prod_autores = pd.concat(frames_prod, ignore_index=True)
top_colaboracion_internacional = (
        df_prod_autores.groupby('Scopus author ID', as_index=False)
        .agg(
            IndiceColaboracionInternacional=('Indice de colaboración internacional', 'mean'), #el promedio en los anos de la productividad del autor
            Citas=('Número de citas en el año', 'sum'),
        )
        .sort_values('IndiceColaboracionInternacional', ascending=False)
        .head(5)
    )
top_colaboracion_internacional['IndiceColaboracionInternacional'] = (
top_colaboracion_internacional['IndiceColaboracionInternacional'].round(3)
)
top_colaboracion_internacional




,Scopus author ID,IndiceColaboracionInternacional,Citas
2118,57193816960,0.980,810.0
1491,56509988200,0.979,12136.0
5500,7004508954,0.975,2062.0
3303,57216482927,0.974,126656.0
662,54402559700,0.949,724.0


In [ ]:
top_colaboracion_nacional = (
        df_prod_autores.groupby('Scopus author ID', as_index=False)
        .agg(
            IndiceColaboracionNacional=('Indice de colaboración nacional', 'mean'), #el promedio en los anos de la productividad del autor
            Citas=('Número de citas en el año', 'sum'),
        )
        .sort_values('IndiceColaboracionNacional', ascending=False)
        .head(5)
        )
top_colaboracion_nacional['IndiceColaboracionNacional'] = (
top_colaboracion_nacional['IndiceColaboracionNacional'].round(3)
)
top_colaboracion_nacional

,Scopus author ID,IndiceColaboracionNacional,Citas
5491,7004260997,0.942,3399.0
2721,57204530210,0.931,80.0
3166,57212810205,0.929,33.0
4358,58020252900,0.929,10.0
779,55399527300,0.924,6147.0


## Top autores con más indice de publicaciones SOLO UAM e indice de publicaciones personal

In [35]:
if not df_prod_autores.empty:
    if 'Indice de publicaciones SOLO UAM' in df_prod_autores.columns:
        top_publicaciones_solo_uam = (
            df_prod_autores.groupby('Scopus author ID', as_index=False)
            .agg(
                IndicePublicacionesSoloUAM=('Indice de publicaciones SOLO UAM', 'mean'),
                Citas=('Número de citas en el año', 'sum'),
            )
            .sort_values('IndicePublicacionesSoloUAM', ascending=False)
            .head(5)
        )
        top_publicaciones_solo_uam['IndicePublicacionesSoloUAM'] = (
            top_publicaciones_solo_uam['IndicePublicacionesSoloUAM'].round(3)
        )
        top_publicaciones_solo_uam
    else:
        print('La columna "Indice de publicaciones SOLO UAM" no existe en los archivos de productividad.')

    if 'Indice de publicaciones personal' in df_prod_autores.columns:
        top_publicaciones_personal = (
            df_prod_autores.groupby('Scopus author ID', as_index=False)
            .agg(
                IndicePublicacionesPersonal=('Indice de publicaciones personal', 'mean'),
                Citas=('Número de citas en el año', 'sum'),
            )
            .sort_values('IndicePublicacionesPersonal', ascending=False)
            .head(5)
        )
        top_publicaciones_personal['IndicePublicacionesPersonal'] = (
            top_publicaciones_personal['IndicePublicacionesPersonal'].round(3)
        )
        top_publicaciones_personal
    else:
        print('La columna "Indice de publicaciones personal" no existe en los archivos de productividad.')
else:
    print('No hay datos de productividad de autores cargados para calcular estos índices.')


In [38]:
top_publicaciones_solo_uam


,Scopus author ID,IndicePublicacionesSoloUAM,Citas
5664,8912885700,1.0,0.0
5673,9435298500,1.0,0.0
5657,8706594300,1.0,80.0
5652,8682163300,1.0,4.0
1764,57188681775,1.0,6.0


In [40]:

top_publicaciones_personal

,Scopus author ID,IndicePublicacionesPersonal,Citas
641,52163448500,1.0,2.0
452,35998860900,1.0,24.0
235,24438327200,1.0,1.0
3982,57359718100,1.0,1.0
2811,57205881047,1.0,3.0


## Autores con más y menos número de autores diferentes

In [43]:
if not df_prod_autores.empty:
    if 'Número de autores diferentes' in df_prod_autores.columns:
        top_autores_mas_autores = (
            df_prod_autores.groupby('Scopus author ID', as_index=False)
            .agg(
                NumeroAutoresDiferentes=('Número de autores diferentes', 'mean'),
            )
            .sort_values('NumeroAutoresDiferentes', ascending=False)
            .head(2)
        )
        top_autores_mas_autores['NumeroAutoresDiferentes'] = top_autores_mas_autores['NumeroAutoresDiferentes'].round(3)

        top_autores_menos_autores = (
            df_prod_autores.groupby('Scopus author ID', as_index=False)
            .agg(
                NumeroAutoresDiferentes=('Número de autores diferentes', 'mean'),
                Citas=('Número de citas en el año', 'sum'),

            )
            .sort_values('NumeroAutoresDiferentes', ascending=True)
            .head(2)
        )
        top_autores_menos_autores['NumeroAutoresDiferentes'] = top_autores_menos_autores['NumeroAutoresDiferentes'].round(3)

        print('Dos autores con mayor número de autores diferentes:')
        display(top_autores_mas_autores)
        print('Dos autores con menor número de autores diferentes:')
        display(top_autores_menos_autores)
    else:
        print('La columna "Número de autores diferentes" no existe en los archivos de productividad.')
else:
    print('No hay datos de productividad de autores cargados para calcular este indicador.')

Dos autores con mayor número de autores diferentes:


,Scopus author ID,NumeroAutoresDiferentes
3303,57216482927,1182.80
1491,56509988200,638.25


Dos autores con menor número de autores diferentes:


,Scopus author ID,NumeroAutoresDiferentes,Citas
3908,57246186500,1.0,2.0
3926,57273473400,1.0,0.0
